In [ ]:
from main import sheet_processor, load_isp_info, determine_sensitivity
from models.sdd_report import SDDReport
import datetime
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os

logger = logging.getLogger(__name__)

# Model selection


In [18]:
MODEL = 'gpt-4.1-nano'

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


In [3]:
# Initalize datasampler
from utils.processing import DataSampler

sampler = DataSampler()

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [4]:
file_path = 'research/data/data.xlsx'  # TODO change
sheets = sampler.sample(file_path)

print(f'Detected sheets: {list(sheets.keys())}')

Detected sheets: ['Sheet1']


# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


In [ ]:
resource_id = None
file_name = None
download_url = None
reports = []

isp = load_isp_info(None, None, file_path.split('/')[-1])

for sheet_name, df in sheets.items():
    sdd_report = SDDReport(
        resource_id=resource_id if resource_id else 'local',
        file_name=file_name if file_name else file_path.split('/')[-1],
        file_url=download_url if download_url else file_path,
        sheet_name=sheet_name,
        processing_timestamp=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        processing_success=True,
        n_records=len(df),
        n_columns=len(df.columns),
    )
    try:
        sdd_report = sheet_processor(sdd_report, isp, df, model=MODEL)
        reports.append(sdd_report.to_dict())

    except Exception as e:
        logger.error(e)
        sdd_report.processing_success = False
        sdd_report.error_message = str(e)
        reports.append(sdd_report.to_dict())

sensitivity = determine_sensitivity(reports)

Classifying PII:   0%|          | 0/8 [00:00<?, ?it/s]

[14448 - 8687722688] 2025-11-30 22:42:28,105 INFO  [httpx:1025] HTTP Request: POST https://hdx-azurellm-classification.cognitiveservices.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Classifying PII:  12%|█▎        | 1/8 [00:00<00:05,  1.21it/s]

[14448 - 8687722688] 2025-11-30 22:42:28,436 INFO  [httpx:1025] HTTP Request: POST https://hdx-azurellm-classification.cognitiveservices.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Classifying PII:  25%|██▌       | 2/8 [00:01<00:03,  1.89it/s]

...

[14448 - 8687722688] 2025-11-30 22:42:30,220 INFO  [httpx:1025] HTTP Request: POST https://hdx-azurellm-classification.cognitiveservices.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Reflecting on PII sensitivity:  12%|█▎        | 1/8 [00:00<00:05,  1.24it/s]

[14448 - 8687722688] 2025-11-30 22:42:30,524 INFO  [httpx:1025] HTTP Request: POST https://hdx-azurellm-classification.cognitiveservices.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Reflecting on PII sensitivity: 100%|██████████| 8/8 [00:01<00:00,  7.23it/s]


[14448 - 8687722688] 2025-11-30 22:42:32,677 INFO  [httpx:1025] HTTP Request: POST https://hdx-azurellm-classification.cognitiveservices.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


# Save


In [ ]:
# Save the report in the research/results/test_results folder under the corresponding file name
folder = f'research/results/test_results/{MODEL}'
# Check if folder exists, if not create it
if not os.path.exists(folder):
    os.makedirs(folder)

file_name = f'{file_path.split("/")[-1]}.json'
save_path = os.path.join(folder, file_name)
with open(save_path, 'w') as f:
    json.dump(reports, f, indent=2)

In [7]:
print(f'Report saved to {save_path}')

Report saved to research/results/test_results/gpt-4.1-nano/data.xlsx.json


# Evaluation


In [13]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [17]:
filename = 'data.xlsx'

# Read the groundtruth and predictions
with open(f'research/results/test_results/groundtruth/{filename}.json', 'r') as f:
    groundtruth = json.load(f)
with open(f'research/results/test_results/gpt-4.1-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth, predictions)),
    }
}
metrics

{'data.xlsx': {'pii_columns': {'accuracy': 0.875,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0},
  'pii_table_level': {'accuracy': 0.0,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0},
  'non_pii_table_level': {'accuracy': 0.0,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0}}}